In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import numpy as np
import matplotlib
matplotlib.use('Agg') # 后台绘图
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import time

# ================= 配置区域 =================
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# [修改] 对应 STFT 张量数据集路径
DATASET_ROOT = "./dataset_STFT_Tensor" 
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

# [修改] 结果输出目录
RESULTS_DIR = "./results_stft_full"

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]
NUM_CLASSES = len(CLASS_NAMES)
BATCH_SIZE = 128
LEARNING_RATE = 0.1
NUM_EPOCHS = 100
PATIENCE = 15 # 早停耐心值 (Epochs)

os.makedirs(RESULTS_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_model_stft.pth")

# ================= 1. 数据集定义 =================
class InMemoryDataset(Dataset):
    def __init__(self, root_dir):
        self.data = []
        self.labels = []
        
        if not os.path.exists(root_dir):
            print(f"[Warn] Directory not found: {root_dir}")
            return
            
        print(f"Loading {root_dir}...")
        files = []
        for cls_name in CLASS_NAMES:
            d = Path(root_dir) / cls_name
            if d.exists(): 
                files.extend([(p, CLASS_NAMES.index(cls_name)) for p in d.glob("*.pt")])
        
        # 使用进度条加载
        with tqdm(total=len(files), unit="file") as pbar:
            for p, label in files:
                try:
                    # 加载 .pt 张量 (3, 64, 64)
                    tensor = torch.load(p)
                    self.data.append(tensor) 
                    self.labels.append(label)
                except Exception as e:
                    pass
                pbar.update(1)

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx]

# ================= 2. 模型定义 (含 Channel Dropout) =================
class ChannelDropout(nn.Module):
    def __init__(self, drop_prob=0.3): # 降低一点概率，防止欠拟合
        super(ChannelDropout, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training: return x
        
        # x: (Batch, 3, H, W)
        batch, channel, _, _ = x.shape
        keep_mask = torch.bernoulli(torch.full((batch, channel, 1, 1), 1 - self.drop_prob, device=x.device))
        
        # 保护机制: 至少保留一个通道
        mask_sum = keep_mask.sum(dim=1, keepdim=True)
        keep_mask = torch.where(mask_sum == 0, torch.ones_like(keep_mask), keep_mask)
        
        return x * keep_mask / (1 - self.drop_prob)

class STFT_CNN(nn.Module):
    def __init__(self, num_classes):
        super(STFT_CNN, self).__init__()
        
        self.channel_drop = ChannelDropout(drop_prob=0.3)
        
        self.features = nn.Sequential(
            # Input: (3, 64, 64) -> Amp, Phase, DiffPhase
            nn.Conv2d(3, 32, 3, padding=1), 
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2), # -> (32, 32, 32)
            nn.Dropout2d(0.1),
            
            nn.Conv2d(32, 64, 3, padding=1), 
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), # -> (64, 16, 16)
            nn.Dropout2d(0.1),
            
            nn.Conv2d(64, 128, 3, padding=1), 
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)  # -> (128, 8, 8)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*8*8, 256),
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.channel_drop(x)
        x = self.features(x)
        x = self.classifier(x)
        return x

# ================= 3. 辅助函数 =================
def plot_history(history):
    epochs = range(1, len(history['train_loss']) + 1)
    
    plt.figure(figsize=(14, 6))
    
    # Loss Curve
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'r-', label='Val Loss')
    plt.title('Loss Curve')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Acc Curve
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    plt.plot(epochs, history['val_acc'], 'r-', label='Val Acc')
    plt.title('Accuracy Curve')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "training_history.png"))
    plt.close()

def plot_cm(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    # 动态获取存在的类别，避免空白行列
    indices = sorted(list(set(y_true) | set(y_pred)))
    names = [CLASS_NAMES[i] for i in indices]
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=names, yticklabels=names)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, filename))
    plt.close()

# ================= 4. 主流程 =================
def main():
    print("=== Training STFT-Tensor Model (Full Features) ===")
    
    # 1. 加载数据
    train_ds = InMemoryDataset(SIM_TRAIN_DIR)
    val_ds = InMemoryDataset(SIM_VAL_DIR)
    real_ds = InMemoryDataset(REAL_TEST_DIR)
    
    if len(train_ds) == 0:
        print("Error: No training data found.")
        return

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # 2. 初始化
    model = STFT_CNN(NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)
    criterion = nn.CrossEntropyLoss()
    
    # 3. 训练循环 (含早停)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    epochs_no_improve = 0
    
    print(f"\nStarting training for {NUM_EPOCHS} epochs (Patience={PATIENCE})...")
    
    for epoch in range(NUM_EPOCHS):
        start_time = time.time()
        
        # --- Train ---
        model.train()
        r_loss, correct, total = 0.0, 0, 0
        
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            r_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            
        train_loss = r_loss / total
        train_acc = correct / total
        
        # --- Val ---
        model.eval()
        v_loss_sum, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                v_loss_sum += criterion(outputs, labels).item() * imgs.size(0)
                _, preds = outputs.max(1)
                v_correct += preds.eq(labels).sum().item()
                v_total += labels.size(0)
                
        val_loss = v_loss_sum / v_total
        val_acc = v_correct / v_total
        
        # 记录
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # 学习率调整
        scheduler.step(val_acc)
        
        duration = time.time() - start_time
        print(f"Epoch {epoch+1:03d} | Time: {duration:.1f}s | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
        
        # 早停逻辑
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print("  >>> Best Model Saved <<<")
        else:
            epochs_no_improve += 1
            print(f"  No improvement for {epochs_no_improve} epochs.")
            if epochs_no_improve >= PATIENCE:
                print("\nEarly Stopping Triggered!")
                break
                
    # 绘图
    plot_history(history)
    
    # 4. 真实数据测试
    print("\nLoading Best Model for Real Data Testing...")
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    model.eval()
    
    if len(real_ds) > 0:
        real_loader = DataLoader(real_ds, batch_size=BATCH_SIZE)
        y_true, y_pred = [], []
        
        print("Running Inference on Real Data...")
        with torch.no_grad():
            for imgs, labels in real_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                _, preds = model(imgs).max(1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())
        
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        acc = np.mean(y_true == y_pred)
        print(f"\nFinal Real Data Accuracy: {acc:.4f} ({acc*100:.2f}%)")
        
        plot_cm(y_true, y_pred, 
                f"Real Data Confusion Matrix (Acc: {acc:.2%})", 
                "real_confusion_matrix.png")
    else:
        print("No real data found for testing.")

if __name__ == "__main__":
    main()

=== Training STFT-Tensor Model (Full Features) ===
Loading ./dataset_STFT_Tensor\sim_train...


100%|██████████| 9603/9603 [00:00<00:00, 11211.78file/s]


Loading ./dataset_STFT_Tensor\sim_val...


100%|██████████| 2397/2397 [00:00<00:00, 11275.43file/s]


Loading ./dataset_STFT_Tensor\real_test...


100%|██████████| 4500/4500 [00:00<00:00, 10922.41file/s]



Starting training for 100 epochs (Patience=15)...
Epoch 001 | Time: 3.2s | Train Loss: 56.6148 Acc: 0.0870 | Val Loss: 2.4872 Acc: 0.0776
  >>> Best Model Saved <<<
Epoch 002 | Time: 2.9s | Train Loss: 2.4923 Acc: 0.0909 | Val Loss: 2.4896 Acc: 0.0855
  >>> Best Model Saved <<<
Epoch 003 | Time: 2.8s | Train Loss: 2.4917 Acc: 0.0829 | Val Loss: 2.4920 Acc: 0.0776
  No improvement for 1 epochs.
Epoch 004 | Time: 2.8s | Train Loss: 2.4918 Acc: 0.0817 | Val Loss: 2.4882 Acc: 0.0922
  >>> Best Model Saved <<<
Epoch 005 | Time: 3.1s | Train Loss: 2.4914 Acc: 0.0848 | Val Loss: 2.4949 Acc: 0.0788
  No improvement for 1 epochs.
Epoch 006 | Time: 3.1s | Train Loss: 2.4920 Acc: 0.0828 | Val Loss: 2.4923 Acc: 0.0776
  No improvement for 2 epochs.
Epoch 007 | Time: 3.1s | Train Loss: 2.4913 Acc: 0.0833 | Val Loss: 2.5030 Acc: 0.0776
  No improvement for 3 epochs.
Epoch 008 | Time: 3.1s | Train Loss: 2.4932 Acc: 0.0828 | Val Loss: 2.4888 Acc: 0.0897
  No improvement for 4 epochs.
Epoch 009 | Time